In [5]:
import torch
import json
import numpy as np
import pretty_midi
from model import TransformerXL  # Ensure this file exists in your directory

def tokenize_midi_with_note_on_off(data):
    tokens = []
    event_signatures = []

    for entry in data:
        composer = f"COMPOSER_{entry['composer']}"
        tokens.append(composer)

        events = []
        
        for event in entry['events']:
            pitch = f"PITCH_{event['pitch']}"
            velocity = f"VEL_{event['velocity']}"
            instrument = f"INST_{event['instrument']}"
            
            # Note On Event
            events.append({
                "type": "NOTE_ON",
                "start": event["start"],
                "pitch": pitch,
                "velocity": velocity,
                "instrument": instrument
            })

            # Note Off Event
            events.append({
                "type": "NOTE_OFF",
                "end": event["end"],
                "pitch": pitch,
                "velocity": velocity,
                "instrument": instrument
            })

        # Sort by time (NOTE_ON by start, NOTE_OFF by end)
        events.sort(key=lambda x: x.get("start", x.get("end", 0)))

        for e in events:
            if e["type"] == "NOTE_ON":
                token = (f"NOTE_ON_{e['pitch']}", e["velocity"], e["instrument"], f"START_{e['start']}")
            else:
                token = (f"NOTE_OFF_{e['pitch']}", e["velocity"], e["instrument"], f"END_{e['end']}")
            
            tokens.append(token)
            event_signatures.append(tuple(sorted(token)))

    # Create unique mapping
    unique_events = list(set(event_signatures))
    event_to_token = {str(event): idx for idx, event in enumerate(unique_events)}
    
    # Generate tokenized data
    tokenized_events = [event_to_token[str(event)] for event in event_signatures]
    
    # Save mappings
    with open("tokens.json", "w") as f:
        json.dump(tokens, f, indent=4)
    
    with open("tokenized_events.json", "w") as f:
        json.dump(tokenized_events, f, indent=4)
    
    with open("event_to_token.json", "w") as f:
        json.dump(event_to_token, f, indent=4)
    
    return tokens, tokenized_events, event_to_token

def load_model(model_path, vocab_size, time_vocab_size, device):
    print(f"Initializing model on {device}...")
    model = TransformerXL(vocab_size, time_vocab_size).to(device)
    
    try:
        print(f"Loading model weights from {model_path}...")
        # Use weights_only=False since we trust the source
        state_dict = torch.load(model_path, map_location=device, weights_only=False)
        model.load_state_dict(state_dict, strict=False)
        print("\u2705 Model loaded successfully!")
    except Exception as e:
        print(f"\u274C Error loading model: {e}")
        return None

    model.eval()
    return model

def process_midi_to_data(midi_path):
    """Convert MIDI file to the data format expected by tokenizer"""
    try:
        midi = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        print(f"\u274C Error loading MIDI file: {e}")
        return None

    data = []
    events = []
    
    for instrument in midi.instruments:
        for note in instrument.notes:
            event = {
                'pitch': note.pitch,
                'velocity': note.velocity,
                'instrument': instrument.program,
                'start': float(note.start),
                'end': float(note.end)
            }
            events.append(event)
    
    # Sort events by start time
    events.sort(key=lambda x: x['start'])
    
    # Create entry
    entry = {
        'composer': 'unknown',
        'events': events
    }
    
    data.append(entry)
    return data

def generate_music(model, seed_sequence, max_length=200, device="cuda"):
    print("Generating sequence using GPU...")
    model.eval()
    generated = seed_sequence[:]
    
    # Move to GPU and convert to tensor
    input_seq = torch.tensor([seed_sequence], dtype=torch.long, device=device)
    
    # Batch process for efficiency
    batch_size = 1
    with torch.no_grad():
        for i in range(max_length):
            if i % 50 == 0:  # Progress update
                print(f"Generating token {i}/{max_length}...")
                if device == "cuda":
                    print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**2:.2f}MB")
            
            # Generate next token
            output = model(input_seq)
            next_token = torch.argmax(output[:, -1, :], dim=-1).item()
            generated.append(next_token)
            
            # Update input sequence
            input_seq = torch.cat((input_seq, torch.tensor([[next_token]], device=device)), dim=1)
            
            # Optional: Clear GPU cache periodically
            if device == "cuda" and i % 100 == 0:
                torch.cuda.empty_cache()
    
    print("Generation complete!")
    return generated

def tokens_to_midi(tokens, token_to_event, output_midi_path):
    midi = pretty_midi.PrettyMIDI()
    current_notes = {}  # Track active notes
    instruments = {}  # Cache for instruments
    
    for token in tokens:
        event_str = token_to_event.get(str(token))
        if not event_str:
            continue
            
        try:
            event = eval(event_str)  # Safe as we control the token_to_event mapping
        except:
            continue
            
        if "NOTE_ON" in event[0]:
            pitch = int(event[0].split('_')[2])
            velocity = int(event[1].split('_')[1])
            instrument_num = int(event[2].split('_')[1])
            start_time = float(event[3].split('_')[1])
            
            current_notes[pitch] = {
                'start': start_time,
                'velocity': velocity,
                'instrument': instrument_num
            }
            
        elif "NOTE_OFF" in event[0]:
            pitch = int(event[0].split('_')[2])
            end_time = float(event[3].split('_')[1])
            
            if pitch in current_notes:
                note_data = current_notes[pitch]
                
                # Get or create instrument
                program = note_data['instrument']
                if program not in instruments:
                    instrument = pretty_midi.Instrument(program=program)
                    midi.instruments.append(instrument)
                    instruments[program] = instrument
                
                # Create note
                note = pretty_midi.Note(
                    velocity=note_data['velocity'],
                    pitch=pitch,
                    start=note_data['start'],
                    end=end_time
                )
                instruments[program].notes.append(note)
                del current_notes[pitch]
    
    # Sort notes in each instrument
    for instrument in midi.instruments:
        instrument.notes.sort(key=lambda x: x.start)
    
    midi.write(output_midi_path)
    print(f"\u2705 MIDI file saved: {output_midi_path}")

def generate(midi_input="chopin.mid", model_path="transformer_xl_music_tpu.pth", output_midi="generated_music.mid"):
    """
    Main function to generate music. Can be called directly from notebook or command line.
    """
    # Setup GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"Initial GPU Memory:")
        print(f"Allocated: {torch.cuda.memory_allocated()/1024**2:.2f}MB")
        print(f"Cached: {torch.cuda.memory_reserved()/1024**2:.2f}MB")
        torch.cuda.empty_cache()
    else:
        print("No GPU available, using CPU")
    
    # Process MIDI to data format
    print("Processing input MIDI...")
    data = process_midi_to_data(midi_input)
    if data is None:
        return
    
    # Tokenize the data
    print("Tokenizing data...")
    tokens, tokenized_events, event_to_token = tokenize_midi_with_note_on_off(data)
    
    # Load model
    print("Loading model...")
    model = load_model(model_path, len(event_to_token), len(event_to_token), device)
    if model is None:
        return
    
    # Generate music
    print("Generating music...")
    generated_sequence = generate_music(model, tokenized_events, max_length=500, device=device)
    
    # Convert back to MIDI
    print("Converting to MIDI...")
    with open("event_to_token.json") as f:
        event_to_token = json.load(f)
    token_to_event = {v: k for k, v in event_to_token.items()}
    
    tokens_to_midi(generated_sequence, token_to_event, output_midi)
    print(f"\u2705 Music generation complete! Output: {output_midi}")
    
    # Final GPU stats
    if device == "cuda":
        print(f"\nFinal GPU Memory Usage:")
        print(f"Allocated: {torch.cuda.memory_allocated()/1024**2:.2f}MB")
        print(f"Cached: {torch.cuda.memory_reserved()/1024**2:.2f}MB")
        torch.cuda.empty_cache()

if __name__ == "__main__":
    import sys
    if not any(x in sys.argv[0] for x in ['ipykernel_launcher.py', 'jupyter-notebook']):
        import argparse
        parser = argparse.ArgumentParser(description='Generate music using TransformerXL model')
        parser.add_argument('--input', default="input.mid", help='Input MIDI file path')
        parser.add_argument('--model', default="transformer_xl_music_tpu.pth", help='Model checkpoint path')
        parser.add_argument('--output', default="generated_music.mid", help='Output MIDI file path')
        
        args = parser.parse_args()
        generate(args.input, args.model, args.output)
    else:
        generate()

No GPU available, using CPU
Processing input MIDI...
Tokenizing data...
Loading model...
Initializing model on cpu...
Loading model weights from transformer_xl_music_tpu.pth...
❌ Error loading model: Error(s) in loading state_dict for TransformerXL:
	size mismatch for time_embedding.weight: copying a param with shape torch.Size([13751, 384]) from checkpoint, the shape in current model is torch.Size([172, 512]).
